In [1]:
import numpy as np
import cv2 
import pandas as pd
import yaml

# 设置Numpy的打印选项
# 精确位数3，不启用科学计数法
np.set_printoptions(precision=3, suppress=True)

In [2]:
# 载入九点标定法的配置文件
handeye_9points_config = None
with open("config/handeye_calibration/handeye_9points.yaml", 'r', encoding='utf-8') as f:
	handeye_9points_config = yaml.load(f.read(), Loader=yaml.SafeLoader)

In [3]:
# 新的相机内参矩阵(畸变去除后)
M_intrisic_new = np.loadtxt('config/usb_camera/M_intrisic_new.txt', delimiter=',')
# 因为是按畸变去除后的图像来计算，因此畸变系数为零向量
distor_coeff = np.float64([0, 0, 0, 0, 0]).astype("float32")

In [4]:
# 载入九点在2D图像中的坐标
img_9points = np.loadtxt("config/handeye_calibration/img9points.txt", delimiter=',')
print(f"9点在像素坐标系的坐标: \n{img_9points}")


9点在像素坐标系的坐标: 
[[ 225.  147.]
 [ 667.  141.]
 [1099.  129.]
 [ 180.  401.]
 [ 678.  396.]
 [1165.  377.]
 [ 121.  716.]
 [ 690.  705.]
 [1248.  680.]]


In [5]:
# 9点在工作台坐标系的坐标
# ws_9points_df = pd.read_excel("config/九点坐标-工作台坐标系.xlsx")
# ws_9points = np.float64(ws_9points_df.to_numpy())[:, 1:]
w = handeye_9points_config["board_width"]
h = handeye_9points_config["board_height"]
# P0点的坐标
x0 = 0.5 * h
y0 = 0.5 * w
ws_9points = np.float64([
	[x0, y0, 0], 	#P0
	[x0, 0, 0], 	#P1
	[x0, -y0, 0], 	#P2
 	[0, y0, 0], 	#P3
	[0, 0, 0], 		#P4
	[0, -y0, 0], 	#P5
	[-x0, y0, 0], 	#P6
	[-x0, 0, 0], 	#P7
	[-x0, -y0, 0], 	#P8
])
print(f"9点在工作台坐标系的坐标: \n{ws_9points}")

9点在工作台坐标系的坐标: 
[[ 45.  65.   0.]
 [ 45.   0.   0.]
 [ 45. -65.   0.]
 [  0.  65.   0.]
 [  0.   0.   0.]
 [  0. -65.   0.]
 [-45.  65.   0.]
 [-45.   0.   0.]
 [-45. -65.   0.]]


In [6]:
# 用solvePnPRansac会更稳一些
ret, rvec, tvec, *_ = cv2.solvePnPRansac(ws_9points, img_9points, \
                    M_intrisic_new, distor_coeff,  flags=cv2.SOLVEPNP_ITERATIVE)

In [7]:
# 构造相机坐标系到工作台坐标系的变换矩阵
T_cam2ws = np.eye(4)
T_cam2ws[:3, :3] = cv2.Rodrigues(rvec)[0]
T_cam2ws[:3, 3] = tvec.reshape(-1)
print("T_cam2ws")
print(T_cam2ws)
np.savetxt("config/handeye_calibration/T_cam2ws.txt", T_cam2ws, fmt='%.3f', delimiter=",")

T_cam2ws
[[ -0.036  -0.999  -0.013  -2.619]
 [ -0.839   0.024   0.544 -18.207]
 [ -0.544   0.03   -0.839 189.405]
 [  0.      0.      0.      1.   ]]
